# Lab 22-05 · Magnetics Design

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import magnetics as mg

AE, LE, AN, MLT, AL0 = 97.1e-6, 78.6e-3, 122e-6, 60.5e-3, 2600e-9   # TDK ETD 34/17/11, N87
MUE0 = AL0 * LE / (mg.MU0 * AE)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Saturation of the flyback core

*Predict* the current at 5.61 µs, the time at which B reaches 390 mT, and the current at 12.5 µs if L stayed 223 µH.

In [2]:
N, G, BS = 32, mg.gap_for_inductance(223e-6, 32, AE, LE, MUE0), 0.39
print(f"gap {G*1e3:.4f} mm, time to 390 mT {N*AE*BS/100*1e6:.2f} us, linear current at 12.5 us {100*12.5e-6/223e-6:.2f} A")


def current(t):
    """Same physics as saturation.cir: B = lam/(N Ae); B = Bs tanh(mur mu0 H/Bs) + mu0 H; N i = H le + B g/mu0."""
    b = 100 * t / (N * AE)
    h = optimize.brentq(lambda x: BS * math.tanh(MUE0 * mg.MU0 * x / BS) + mg.MU0 * x - b, 0, 1e7)
    return (h * LE + b * G / mg.MU0) / N


t = np.linspace(0, 12.6e-6, 600)
plt.figure()
plt.plot(t * 1e6, [current(x) for x in t], label="saturating core")
plt.plot(t * 1e6, 100 * t / 223e-6, ls=":", label="linear 223 uH")
plt.axvline(N * AE * BS / 100 * 1e6, color="0.5", ls="--", lw=0.8)
plt.ylim(0, 30); plt.xlabel("time (us)"); plt.ylabel("winding current (A)"); plt.legend(); plt.show()
s = ngspice("saturation.cir")
print({k: s[k] for k in ("i_ton", "i_115", "i_125")}, "algebra:", [round(current(x), 3) for x in (5.61e-6, 11.5e-6, 12.5e-6)])

gap 0.5134 mm, time to 390 mT 12.12 us, linear current at 12.5 us 5.61 A
{'i_ton': 2.532867, 'i_115': 5.54921, 'i_125': 29.16102} algebra: [2.533, 5.549, 29.161]


## Part 2 · Leakage: layered against interleaved

*Predict* the leakage of Example 22-05.8's 9-turn transformer both ways.

In [3]:
for name, fn in (("layered", mg.leakage_inductance), ("interleaved", mg.leakage_interleaved)):
    print(f"{name}: {fn(9, MLT, 20.9e-3, 0.6e-3, 0.6e-3, 0.1e-3)*1e6:.4f} uH")
s = ngspice("leakage.cir")
print({k: s[k] for k in ("lsc_a", "lsc_b", "loc")})

layered: 0.1473 uH
interleaved: 0.0442 uH
{'lsc_a': 1.472982e-07, 'lsc_b': 4.419468e-08, 'loc': 0.0002106737}


## Part 3 · Dowell's factor and the optimum foil

*Predict* how F_R grows with Δ for 1 to 5 layers, and the foil thickness that minimizes Example 22-05.6's four-layer loss.

In [4]:
d = np.linspace(0.05, 3, 300)
plt.figure()
for m in (1, 2, 3, 5):
    plt.semilogy(d, [mg.dowell_fr(x, m) for x in d], label=f"M = {m}")
plt.xlabel("Delta = h/delta"); plt.ylabel("F_R"); plt.legend(); plt.show()
rho = mg.rho_cu(100)
delta = mg.skin_depth(100e3, rho)
loss = lambda h: 20 ** 2 * mg.winding_resistance(4, MLT, h * 20e-3, rho) * mg.dowell_fr(h / delta, 4)
h = np.linspace(0.03e-3, 0.5e-3, 300)
plt.figure(); plt.plot(h * 1e3, [loss(x) for x in h]); plt.xlabel("foil thickness (mm)"); plt.ylabel("loss (W)"); plt.show()
r = optimize.minimize_scalar(loss, bounds=(0.03e-3, 0.5e-3), method="bounded", options={"xatol": 1e-9})
print(f"optimum {r.x*1e3:.3f} mm (Delta {r.x/delta:.2f}), loss {r.fun:.3f} W")

optimum 0.159 mm (Delta 0.66), loss 0.923 W


## Part 4 · The flux-swing optimum and a powder core's roll-off

*Predict* Example 22-05.8's optimum ΔB and the cost of 9 turns; then the 60µ Kool Mµ permeability at 50 and 100 Oe.

In [5]:
beta, b0 = 2.253, 0.2
a, b = 1.2 / b0 ** beta, 0.3 * b0 ** 2
bopt = mg.optimum_flux_swing(a, beta, b)
x = np.linspace(0.06, 0.25, 300)
plt.figure()
plt.plot(x, a * x ** beta, label="core"); plt.plot(x, b / x ** 2, label="copper"); plt.plot(x, a * x ** beta + b / x ** 2, label="total")
plt.axvline(bopt, color="0.5", ls=":"); plt.ylim(0, 3); plt.xlabel("flux swing (T)"); plt.ylabel("loss (W)"); plt.legend(); plt.show()
print(f"optimum {bopt:.3f} T with {6*b0/bopt:.2f} turns; 9 turns: {a*(b0*6/9)**beta + b/(b0*6/9)**2:.3f} W vs {a*bopt**beta + b/bopt**2:.3f} W")
kool60 = lambda oe: 100 / (0.01 + 2.142e-6 * oe ** 1.855) / 100   # % of initial permeability (Magnetics fit)
for oe in (50, 100):
    print(f"60u Kool Mu at {oe} Oe ({oe*1000/(4*math.pi)/1e3:.2f} kA/m): {kool60(oe):.1f} % of initial permeability")

optimum 0.140 T with 8.55 turns; 9 turns: 1.156 W vs 1.149 W
60u Kool Mu at 50 Oe (3.98 kA/m): 76.7 % of initial permeability
60u Kool Mu at 100 Oe (7.96 kA/m): 47.7 % of initial permeability


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.